# create Raster from CSV containing data about SSA countries

In [ ]:
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from pathlib import Path
import urllib.request

"""
requisites
- csv must have a column with ISO3 country codes (e.g., "country") and one or more columns with data values
- edit the code to specify the correct column name for ISO3 codes and the path to the CSV file
- Set fill_missing_with_average to True/False to control the filling of missing SSA data
"""

# Modify these paths and the country column name as needed
csv_path = r"C:/Users/Fra/OneDrive - Politecnico di Milano/File di Mattia Cinchetti - Thesis_onstoveM&F/OnStoveThesis/thesis/script_nostri/dataset_first_step/lpg_domestic_shares.csv"
output_path = r"C:/Users/Fra/OneDrive - Politecnico di Milano/File di Mattia Cinchetti - Thesis_onstoveM&F/OnStoveThesis/thesis/script_nostri/dataset_first_step/lpg_domestic_shares.tif"
csv_iso_column = "country" # Name of the column with ISO3 codes in the CSV

# Fixed: defined as a dictionary so .keys() and value indexing works later in the script
selected_columns = {
    "percentage_house": "percentage_house" 
}

# Set this flag to True to calculate and fill missing SSA values, or False to leave them as NoData
fill_missing_with_average = False 

# Default Global Grid (minx, miny, maxx, maxy)
bounds = (-180, -90, 180, 90) 
resolution = 0.1 # Resolution in degrees

# List of Sub-Saharan Africa ISO3 codes
ssa_iso3 = [
    'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CPV', 'CMR', 'CAF', 'TCD', 'COM',
    'COD', 'COG', 'CIV', 'DJI', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB',
    'GHA', 'GIN', 'GNB', 'KEN', 'LSO', 'LBR', 'MDG', 'MWI', 'MLI', 'MRT',
    'MUS', 'MOZ', 'NAM', 'NER', 'NGA', 'RWA', 'STP', 'SEN', 'SYC', 'SLE',
    'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'UGA', 'ZMB', 'ZWE',
]

# Step 0: Load national boundaries (Natural Earth)
print("Loading national boundaries...")
cache_dir = Path.home() / ".geopandas_cache"
cache_dir.mkdir(exist_ok=True)
ne_zip = cache_dir / "ne_10m_admin_0_countries.zip"

if not ne_zip.exists():
    ne_url = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
    urllib.request.urlretrieve(ne_url, ne_zip)

world = gpd.read_file(ne_zip).to_crs("EPSG:4326")

iso_candidates = ["ISO_A3", "ADM0_A3", "SOV_A3", "GU_A3"]
iso_col = next((c for c in iso_candidates if c in world.columns), None)
world["iso3"] = world[iso_col].astype(str).str.strip().str.upper()

# Define raster dimensions
width = int((bounds[2] - bounds[0]) / resolution)
height = int((bounds[3] - bounds[1]) / resolution)
transform = from_bounds(*bounds, width, height)

# Pre-calculate the exact number of pixels for each country
print("Calculating pixel counts per country for weighted averages...")
world['pixel_count'] = 0
for idx, row in world.iterrows():
    if row["geometry"] is not None:
        mask = rasterize(
            [(row["geometry"], 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype=np.uint8,
            all_touched=True
        )
        world.at[idx, 'pixel_count'] = mask.sum()

# Step 1: Load CSV and dynamically identify columns
print(f"Loading data from: {csv_path}")
df = pd.read_csv(csv_path)
df[csv_iso_column] = df[csv_iso_column].astype(str).str.strip().str.upper()

# Identify all columns containing data (excluding the ISO column)
if selected_columns:
    missing_cols = [col for col in selected_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Selected columns not found in CSV: {missing_cols}")
    value_columns = list(selected_columns.keys())
    band_names = [selected_columns[col] for col in value_columns]
else:
    value_columns = [col for col in df.columns if col != csv_iso_column]
    band_names = value_columns.copy()

# Ensure data columns are numeric
for col in value_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Step 2: Map data to spatial geometries
print("Mapping data onto national polygons...")
df_indexed = df.set_index(csv_iso_column)

for col in value_columns:
    world[f"val_{col}"] = world["iso3"].map(df_indexed[col])

# Step 3: Dynamic rasterization and saving
print(f"Creating raster with {len(value_columns)} bands...")
with rasterio.open(
    output_path, "w",
    driver="GTiff",
    height=height, width=width,
    count=len(value_columns),
    dtype=np.float32,
    crs="EPSG:4326",
    transform=transform
) as dst:
    
    for band_idx, (col, band_name) in enumerate(zip(value_columns, band_names), start=1):
        target_col = f"val_{col}"
        
        # --- Handle missing Sub-Saharan Africa values based on the flag ---
        if fill_missing_with_average:
            is_ssa = world["iso3"].isin(ssa_iso3)
            valid_ssa = world[is_ssa & world[target_col].notna()]
            missing_ssa = world[is_ssa & world[target_col].isna()]
            
            if not valid_ssa.empty and not missing_ssa.empty:
                total_valid_pixels = valid_ssa['pixel_count'].sum()
                
                if total_valid_pixels > 0:
                    # Calculate the pixel-weighted average
                    weighted_sum = (valid_ssa[target_col] * valid_ssa['pixel_count']).sum()
                    weighted_avg = weighted_sum / total_valid_pixels
                    
                    # Assign the calculated average to missing SSA countries
                    world.loc[missing_ssa.index, target_col] = weighted_avg
                    print(f" - Band {band_idx}: Filled {len(missing_ssa)} missing SSA countries with pixel-weighted average ({weighted_avg:.4f})")
        
        # On-the-fly rasterization for the current column
        raster_array = rasterize(
            [(row["geometry"], float(row[target_col])) for _, row in world.iterrows()
             if pd.notna(row[target_col]) and row["geometry"] is not None],
            out_shape=(height, width),
            transform=transform,
            fill=np.nan,
            dtype=np.float32,
            all_touched=True
        )
        
        # Write the single band and assign the name (tag)
        dst.write(raster_array, band_idx)
        dst.update_tags(band_idx, long_name=band_name)
        print(f" - Band {band_idx} ({band_name}) rasterized and saved.")

print(f"\nProcessing complete. File saved to: {output_path}")